# layernorm_backward  
`layernorm_backward` 是 **LayerNorm 的反向传播函数**，用于求梯度（手动求导，numpy 实现）。  
![layernorm_backward](../../image/layernorm_backward.png)

In [8]:
import numpy as np

In [9]:
class LayerNorm:
    def __init__(self, dim, eps=1e-5):
        self.gamma = np.ones(dim)
        self.beta = np.zeros(dim)
        self.eps = eps

    def forward(self, x):
        mean = x.mean(axis=-1, keepdims=True)
        var = x.var(axis=-1, keepdims=True)
        x_hat = (x - mean) / np.sqrt(var + self.eps)
        y = self.gamma * x_hat + self.beta
        return y, mean, var, x_hat # 返回中间变量给反向传播用

In [10]:
def layernorm_backward(dy, x, ln):
    print("\n===== 进入 layernorm_backward =====")
    print(f"输入 dy(上游梯度) shape = {dy.shape}")
    print(f"输入 x(前向原始输入) shape = {x.shape}")

    mean = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    std_inv = 1.0 / np.sqrt(var + ln.eps)
    x_hat = (x - mean) * std_inv
    n = x.shape[-1]
    print(f"mean shape={mean.shape}, var shape={var.shape}")
    print(f"std_inv shape={std_inv.shape}, n(归一化维度大小) = {n}")

    grad_gamma = (dy * x_hat).sum(axis=(0, 1))
    grad_beta = dy.sum(axis=(0, 1))
    print(f"\ngrad_gamma shape={grad_gamma.shape}, grad_beta shape={grad_beta.shape}")

    dx_hat = dy * ln.gamma
    print(f"dx_hat (dL/dx_hat) shape={dx_hat.shape}")

    dvar = (dx_hat * (x - mean) * -0.5 * std_inv ** 3).sum(axis=-1, keepdims=True)
    print(f"dvar (dL/dσ²) shape={dvar.shape}")

    dmean = (-dx_hat * std_inv).sum(axis=-1, keepdims=True)
    term_mu = dvar * (-2.0 / n) * (x - mean).sum(axis=-1, keepdims=True)
    dmean += term_mu
    print(f"dmean (dL/dμ) shape={dmean.shape}")

    dx = dx_hat * std_inv + dvar * 2.0 * (x - mean) / n + dmean / n
    print(f"dx (dL/dx，传回上一层梯度) shape={dx.shape}")

    return dx, grad_gamma, grad_beta

In [11]:
if __name__ == "__main__":
    # 构造小输入：batch=2, seq_len=3, embed_dim=4
    batch, seq_len, embed_dim = 2, 3, 4
    x = np.array([
        [[1.0, 2.0, 3.0, 4.0],
         [1.0, 1.0, 1.0, 1.0],
         [0.1, 0.2, 0.3, 0.4]],
        [[5.0, 6.0, 7.0, 8.0],
         [2.0, 2.0, 2.0, 2.0],
         [1.1, 1.2, 1.3, 1.4]]
    ])
    print("===== 前向传播开始 =====")
    print(f"输入x shape={x.shape}")
    ln = LayerNorm(dim=embed_dim, eps=1e-5)
    y, mean, var, x_hat = ln.forward(x)
    print(f"LayerNorm输出 y shape={y.shape}")
    print(f"y = \n{y}")

    # 模拟上游传下来的梯度dy：简单全1梯度（方便验算）
    dy = np.ones_like(y)
    print(f"\n模拟上游梯度 dy shape={dy.shape}")

    dx, grad_gamma, grad_beta = layernorm_backward(dy, x, ln)

    print("\n===== 反向传播结果汇总 =====")
    print(f"dx（输入x的梯度）shape={dx.shape}")
    print(f"grad_gamma: {grad_gamma}")
    print(f"grad_beta: {grad_beta}")

===== 前向传播开始 =====
输入x shape=(2, 3, 4)
LayerNorm输出 y shape=(2, 3, 4)
y = 
[[[-1.34163542 -0.44721181  0.44721181  1.34163542]
  [ 0.          0.          0.          0.        ]
  [-1.34110445 -0.44703482  0.44703482  1.34110445]]

 [[-1.34163542 -0.44721181  0.44721181  1.34163542]
  [ 0.          0.          0.          0.        ]
  [-1.34110445 -0.44703482  0.44703482  1.34110445]]]

模拟上游梯度 dy shape=(2, 3, 4)

===== 进入 layernorm_backward =====
输入 dy(上游梯度) shape = (2, 3, 4)
输入 x(前向原始输入) shape = (2, 3, 4)
mean shape=(2, 3, 1), var shape=(2, 3, 1)
std_inv shape=(2, 3, 1), n(归一化维度大小) = 4

grad_gamma shape=(4,), grad_beta shape=(4,)
dx_hat (dL/dx_hat) shape=(2, 3, 4)
dvar (dL/dσ²) shape=(2, 3, 1)
dmean (dL/dμ) shape=(2, 3, 1)
dx (dL/dx，传回上一层梯度) shape=(2, 3, 4)

===== 反向传播结果汇总 =====
dx（输入x的梯度）shape=(2, 3, 4)
grad_gamma: [-5.36547974 -1.78849325  1.78849325  5.36547974]
grad_beta: [6. 6. 6. 6.]
